In [6]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
#Connect Google Drive
from google.colab import drive
drive.mount('/gdrive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install tensorflow opencv-python matplotlib

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [3]:
import os
print(os.path.getsize("/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals_dataset.zip"))

68490945


In [5]:
import os
os.listdir('/content')
!file /content/drive/MyDrive/Data_science_project/dataset/aquatic_animals_dataset.zip

/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals_dataset.zip: Zip archive data, at least v4.5 to extract, compression method=deflate


unzipping the file using the system unzip

In [ ]:
!unzip /content/aquatic_animals_dataset.zip -d /content/dataset

Archive:  /content/aquatic_animals_dataset.zip
  inflating: /content/dataset/aquatic_animals/crab/image_0.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_1.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_10.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_11.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_12.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_13.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_14.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_15.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_16.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_17.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_18.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_19.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_2.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_20.jpeg  
  inflating: /cont

In [7]:
import zipfile

zip_path = "/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals_dataset.zip"
extract_path = "/content/drive/MyDrive/Data_science_project/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete")

Extraction complete


In [8]:
import os

print(os.listdir("/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals"))

['crab', 'dolphin', 'octopus', 'seahorse', 'seal', 'seaturtle', 'shark', 'starfish']


checking the dataset

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_path = "/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals"

img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 320 images belonging to 8 classes.
Found 80 images belonging to 8 classes.


loading pre ptrained model

In [11]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


building my model

In [12]:
from tensorflow.keras import layers, models

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,984 (9.24 MB)

 Trainable params: 165,000 (644.53 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

compiling the model

In [13]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

training model

In [14]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.5750 - loss: 1.3124 - val_accuracy: 0.7875 - val_loss: 0.6397
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 376ms/step - accuracy: 0.9156 - loss: 0.2808 - val_accuracy: 0.8125 - val_loss: 0.4458
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 388ms/step - accuracy: 0.9750 - loss: 0.1076 - val_accuracy: 0.8250 - val_loss: 0.4886
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 416ms/step - accuracy: 0.9969 - loss: 0.0379 - val_accuracy: 0.8375 - val_loss: 0.5191
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 452ms/step - accuracy: 1.0000 - loss: 0.0206 - val_accuracy: 0.8250 - val_loss: 0.5037


testing model with different image

In [15]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/drive/MyDrive/Data_science_project/test_images/killer_whale.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

prediction = model.predict(img_array)
class_names = list(train_data.class_indices.keys())

print("Prediction:", class_names[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
Prediction: dolphin


In [ ]:
import os

os.makedirs("/content/test_images", exist_ok=True)

In [16]:
from tensorflow.keras.preprocessing import image
import numpy as np
import os

test_folder = "/content/drive/MyDrive/Data_science_project/test_images"

class_names = list(train_data.class_indices.keys())

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction)]

    print(f"{img_name} → {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
starfish.jpg → starfish
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
killer_whale.jpg → dolphin
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
octopus.jpg → octopus


In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import os

test_folder = "/content/test_images"
class_names = list(train_data.class_indices.keys())

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)

    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction)

    print(f"{img_name} → {predicted_class} ({confidence:.2f})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
killer_whale.jpg → dolphin (0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
starfish.jpg → starfish (1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
octopus.jpg → octopus (0.97)


In [17]:
import os
os.path.exists('/content/drive')

True

saving the model

In [18]:
model.save("/content/drive/MyDrive/Data_science_project/models/aquatic_model_v1.keras")